# Лабораторная работа №2

## Тема: Транспортная задача и проверка оптимального плана в Python

> В этой работе мы не уходим в транспортный симплекс и табличные эвристики.
> Фокус: правильно собрать транспортную модель, проверить баланс и получить воспроизводимый план перевозок через `scipy.optimize.linprog`.

### Как читать эту работу

- $x_{ij}$ — это объём перевозки от поставщика $i$ к потребителю $j$.
- Матрица затрат показывает цену одной единицы перевозки по каждому маршруту.
- Если суммарный запас равен суммарному спросу, задача закрытая.
- Если суммы не совпадают, задачу нужно сначала сбалансировать фиктивным узлом.


## 1. Цель работы

Освоить базовый цикл решения транспортной задачи на одном и том же примере двумя связанными способами:

1. Содержательно: понять, кто кому и сколько должен отправить.
2. Математически и программно: записать задачу как ЛП и решить её через `scipy.optimize.linprog`.

## 2. Формируемые умения и навыки

После выполнения работы вы сможете:

1. Переводить логистическую ситуацию в транспортную модель.
2. Проверять баланс запасов и спроса.
3. Объяснять смысл закрытой и открытой транспортной задачи.
4. Вводить фиктивного поставщика или фиктивного потребителя, если задача небалансна.
5. Разворачивать матрицу перевозок в формат, удобный для `linprog`.
6. Проверять допустимость найденного плана по строкам и столбцам.
7. Интерпретировать итоговую стоимость и структуру маршрутов.


## 3. Теоретический минимум (кратко)

Транспортная задача в базовой форме записывается так:

$$
\min Z = \sum_{i=1}^{m}\sum_{j=1}^{n} c_{ij}x_{ij}
$$

при ограничениях

$$
\sum_{j=1}^{n} x_{ij} = a_i, \quad i=1,\dots,m,
$$

$$
\sum_{i=1}^{m} x_{ij} = b_j, \quad j=1,\dots,n,
$$

$$
x_{ij} \ge 0.
$$

Здесь:

- $a_i$ — запас поставщика $i$;
- $b_j$ — спрос потребителя $j$;
- $c_{ij}$ — стоимость единицы перевозки по маршруту $(i, j)$;
- $x_{ij}$ — искомый объём перевозки.

В этой лабораторной основной пример уже сбалансирован. Открытые задачи и фиктивные узлы вынесены в самостоятельные варианты.


## 4. Исходная базовая задача

Три аптечных склада должны отправить лекарства в четыре больницы.

### Запасы складов

| Склад | Доступный объём |
| --- | ---: |
| Склад A | 35 |
| Склад B | 50 |
| Склад C | 40 |

### Спрос больниц

| Больница | Требуемый объём |
| --- | ---: |
| Больница 1 | 20 |
| Больница 2 | 30 |
| Больница 3 | 25 |
| Больница 4 | 50 |

### Матрица затрат на 1 условную единицу груза

| Откуда / Куда | Больница 1 | Больница 2 | Больница 3 | Больница 4 |
| --- | ---: | ---: | ---: | ---: |
| Склад A | 4 | 6 | 8 | 13 |
| Склад B | 5 | 4 | 7 | 9 |
| Склад C | 6 | 3 | 4 | 7 |

Нужно найти такой план перевозок, который полностью закроет спрос больниц и минимизирует суммарную стоимость доставки.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linprog

# Здесь мы просто переносим транспортную таблицу из условия в Python.
# Ожидаемый результат: увидеть запасы, спрос и матрицу стоимостей в удобном виде.
warehouses = ["Склад A", "Склад B", "Склад C"]
hospitals = ["Больница 1", "Больница 2", "Больница 3", "Больница 4"]

supply = np.array([35, 50, 40], dtype=float)
demand = np.array([20, 30, 25, 50], dtype=float)
costs = np.array([
    [4, 6, 8, 13],
    [5, 4, 7, 9],
    [6, 3, 4, 7],
], dtype=float)

print("Суммарный запас:", supply.sum())
print("Суммарный спрос:", demand.sum())
print("Задача сбалансирована:", np.isclose(supply.sum(), demand.sum()))

display(pd.DataFrame({"Запас": supply}, index=warehouses))
display(pd.DataFrame({"Спрос": demand}, index=hospitals))
display(pd.DataFrame(costs, index=warehouses, columns=hospitals))

Суммарный запас: 125.0
Суммарный спрос: 125.0
Задача сбалансирована: True


,Запас
Склад A,35.0
Склад B,50.0
Склад C,40.0


,Спрос
Больница 1,20.0
Больница 2,30.0
Больница 3,25.0
Больница 4,50.0


,Больница 1,Больница 2,Больница 3,Больница 4
Склад A,4.0,6.0,8.0,13.0
Склад B,5.0,4.0,7.0,9.0
Склад C,6.0,3.0,4.0,7.0


## 5. Шаг 1. Формулируем математическую модель

Обозначим через $x_{ij}$ объём перевозки со склада $i$ в больницу $j$.

Тогда целевая функция минимизации суммарной стоимости имеет вид:

$$
\min Z = 4x_{11} + 6x_{12} + 8x_{13} + 13x_{14}
+ 5x_{21} + 4x_{22} + 7x_{23} + 9x_{24}
+ 6x_{31} + 3x_{32} + 4x_{33} + 7x_{34}
$$

Ограничения по складам:

$$
x_{11} + x_{12} + x_{13} + x_{14} = 35,
$$

$$
x_{21} + x_{22} + x_{23} + x_{24} = 50,
$$

$$
x_{31} + x_{32} + x_{33} + x_{34} = 40.
$$

Ограничения по больницам:

$$
x_{11} + x_{21} + x_{31} = 20,
$$

$$
x_{12} + x_{22} + x_{32} = 30,
$$

$$
x_{13} + x_{23} + x_{33} = 25,
$$

$$
x_{14} + x_{24} + x_{34} = 50.
$$

И, конечно, все перевозки неотрицательны:

$$
x_{ij} \ge 0.
$$


## 6. Шаг 2. Переход к формату `linprog`

`linprog` ожидает вектор переменных, поэтому матрицу перевозок нужно развернуть по строкам:

$$
(x_{11}, x_{12}, x_{13}, x_{14}, x_{21}, x_{22}, x_{23}, x_{24}, x_{31}, x_{32}, x_{33}, x_{34}).
$$

Тогда:

- `c = costs.flatten()`;
- строки `A_eq` для складов контролируют суммы по строкам матрицы плана;
- строки `A_eq` для больниц контролируют суммы по столбцам;
- `b_eq` состоит из запасов и спроса;
- `bounds = (0, None)` задают неотрицательность всех перевозок.


In [2]:
# Этот блок превращает красивую транспортную таблицу в формат для linprog.
# Ожидаемый результат: минимальная стоимость, план перевозок и проверки баланса.
m, n = costs.shape

# flatten() делает из матрицы затрат одну длинную строку чисел.
# Именно в таком порядке решатель будет хранить переменные x_ij.
c = costs.flatten()

A_eq = []
b_eq = []

# Ограничения по складам: сумма отправок из каждой строки должна равняться запасу.
for i in range(m):
    row = np.zeros(m * n)
    row[i * n : (i + 1) * n] = 1
    A_eq.append(row)
    b_eq.append(supply[i])

# Ограничения по больницам: сумма входящих поставок в каждый столбец должна равняться спросу.
for j in range(n):
    row = np.zeros(m * n)
    row[j::n] = 1
    A_eq.append(row)
    b_eq.append(demand[j])

A_eq = np.array(A_eq, dtype=float)
b_eq = np.array(b_eq, dtype=float)
bounds = [(0, None)] * (m * n)

# Запускаем решатель для транспортной задачи как для обычного линейного программирования.
result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")

print("Успешность решения:", result.success)
print("Статус:", result.status)
print("Сообщение:", result.message)

if not result.success:
    raise RuntimeError("Решатель не нашёл оптимальное решение.")

# Возвращаем длинный вектор ответа обратно в таблицу m x n.
plan = result.x.reshape(m, n)
plan[np.abs(plan) < 1e-9] = 0.0

plan_df = pd.DataFrame(plan, index=warehouses, columns=hospitals)
row_check = pd.DataFrame({
    "Запас": supply,
    "Сумма по плану": plan.sum(axis=1),
}, index=warehouses)
column_check = pd.DataFrame({
    "Спрос": demand,
    "Сумма по плану": plan.sum(axis=0),
}, index=hospitals)

# Собираем только реально использованные маршруты, чтобы итог читался проще.
route_rows = []
for i, source in enumerate(warehouses):
    for j, target in enumerate(hospitals):
        if plan[i, j] > 1e-9:
            route_rows.append({
                "Откуда": source,
                "Куда": target,
                "Объём": plan[i, j],
                "Стоимость за единицу": costs[i, j],
                "Итоговые затраты": plan[i, j] * costs[i, j],
            })

routes_df = pd.DataFrame(route_rows)

print(f"Минимальная суммарная стоимость: {result.fun:.2f}")
print()
print("Оптимальный план перевозок:")
display(plan_df)
print("Проверка по складам:")
display(row_check)
print("Проверка по больницам:")
display(column_check)
print("Использованные маршруты:")
display(routes_df)

# Эти проверки подтверждают, что план честно закрывает и запасы, и спрос.
assert np.allclose(plan.sum(axis=1), supply)
assert np.allclose(plan.sum(axis=0), demand)

Успешность решения: True
Статус: 0
Сообщение: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Минимальная суммарная стоимость: 750.00

Оптимальный план перевозок:


,Больница 1,Больница 2,Больница 3,Больница 4
Склад A,20.0,15.0,0.0,0.0
Склад B,0.0,15.0,0.0,35.0
Склад C,0.0,0.0,25.0,15.0


Проверка по складам:


,Запас,Сумма по плану
Склад A,35.0,35.0
Склад B,50.0,50.0
Склад C,40.0,40.0


Проверка по больницам:


,Спрос,Сумма по плану
Больница 1,20.0,20.0
Больница 2,30.0,30.0
Больница 3,25.0,25.0
Больница 4,50.0,50.0


Использованные маршруты:


,Откуда,Куда,Объём,Стоимость за единицу,Итоговые затраты
0,Склад A,Больница 1,20.0,4.0,80.0
1,Склад A,Больница 2,15.0,6.0,90.0
2,Склад B,Больница 2,15.0,4.0,60.0
3,Склад B,Больница 4,35.0,9.0,315.0
4,Склад C,Больница 3,25.0,4.0,100.0
5,Склад C,Больница 4,15.0,7.0,105.0


## 7. Как интерпретировать результат

После выполнения кода обязательно проговорите смысл ответа словами.

1. Какие маршруты оказались активными?
2. Есть ли дорогие маршруты, которых solver сумел избежать?
3. Видно ли, какие склады в основном обслуживают какие больницы?
4. Совпадают ли суммы по строкам с запасами, а суммы по столбцам — со спросом?

Именно этот шаг превращает таблицу чисел в содержательный логистический вывод.


## 8. Что должно быть в отчёте

1. Таблица запасов и таблица спроса.
2. Матрица затрат.
3. Проверка, что задача сбалансирована.
4. Линейная постановка через переменные $x_{ij}$.
5. Пояснение, как матрица перевозок разворачивается в вектор для `linprog`.
6. Листинг Python-кода.
7. Итоговая матрица оптимального плана.
8. Проверка допустимости по строкам и столбцам.
9. Минимальная стоимость перевозок.
10. Краткая логистическая интерпретация результата.


## 9. Типичные ошибки

1. Перепутали, где в матрице затрат строки, а где столбцы.
2. Не проверили баланс перед решением задачи.
3. Неверно развернули матрицу перевозок в вектор.
4. Собрали `A_eq` так, что часть переменных выпала из ограничений.
5. Посмотрели только на значение целевой функции и не проверили баланс строк и столбцов.
6. Сделали вывод по стоимости, но не объяснили, какие маршруты формируют оптимальный план.
